In [30]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,SampleID,text,label
0,1,i didnt feel humiliated,sadness
1,2,i can go from feeling so hopeless to so damned...,sadness
2,3,im grabbing a minute to post i feel greedy wrong,anger
3,4,i am ever feeling nostalgic about the fireplac...,love
4,5,i am feeling grouchy,anger


In [31]:
df_train.text[46]

'i lost my special mind but don t worry i m still sane i just wanted you to feel what i felt while reading this book i don t know how many times it was said that sam was special but i can guarantee you it was many more times than what i used in that paragraph did i tell you she was special'

In [32]:
import re

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', "", text)
    text = re.sub(r'[\d+]', "", text)
    return text

preprocess(df_train.text[46])

'i lost my special mind but don t worry i m still sane i just wanted you to feel what i felt while reading this book i don t know how many times it was said that sam was special but i can guarantee you it was many more times than what i used in that paragraph did i tell you she was special'

In [33]:
df_train.text = df_train.text.apply(preprocess)
print("train set done")
df_test.text = df_test.text.apply(preprocess)

df_train.head()

train set done


,SampleID,text,label
0,1,i didnt feel humiliated,sadness
1,2,i can go from feeling so hopeless to so damned...,sadness
2,3,im grabbing a minute to post i feel greedy wrong,anger
3,4,i am ever feeling nostalgic about the fireplac...,love
4,5,i am feeling grouchy,anger


In [34]:
df_train.label.unique()

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [35]:
X = df_train.text
y = df_train.label
X_final = df_test.text

from sklearn.preprocessing import LabelEncoder

enc = LabelEncoder()
y = enc.fit_transform(y)

y[:5]

array([4, 4, 0, 3, 0])

In [36]:
from gensim.models import FastText

sentences = [sentence.split() for sentence in X]

ft_model = FastText(sentences=sentences, vector_size=100, window=5, min_count=2, workers=4, sg=1)

import numpy as np

def vectorize(sentence):
    words = sentence.split()
    words_vecs = [ft_model.wv[word] for word in words]
    if len(words_vecs) == 0:
        return np.zeros(ft_model.vector_size)
    return np.mean(words_vecs, axis=0)

X = np.array([vectorize(sentence) for sentence in X])
X_final = np.array([vectorize(sentence) for sentence in X_final])

X

array([[-0.01327776, -0.10141001, -0.13874292, ..., -0.47858387,
        -0.03594397, -0.4956776 ],
       [-0.05511663,  0.0020085 , -0.1791591 , ..., -0.4652311 ,
        -0.07471099, -0.4208973 ],
       [-0.06688221, -0.02436329, -0.17052647, ..., -0.34240514,
        -0.06930987, -0.4278441 ],
       ...,
       [-0.07557572, -0.07298566, -0.12345165, ..., -0.37062332,
        -0.11584502, -0.42208743],
       [-0.01862888, -0.03822101, -0.15045123, ..., -0.3466365 ,
        -0.05833558, -0.39614588],
       [-0.05785612, -0.09015345, -0.14343104, ..., -0.37437487,
        -0.08305331, -0.36604136]], dtype=float32)

In [37]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score

model = LinearSVC()

cross_val_score(model, X, y, cv=2, scoring='f1_weighted')

array([0.3517397 , 0.34346206])

In [27]:
model.fit(X, y)
y_final = model.predict(X_final)
y_final[:5]

array([4, 2, 4, 2, 4])

In [28]:
y_final = enc.inverse_transform(y_final)
y_final[:5]

array(['sadness', 'joy', 'sadness', 'joy', 'sadness'], dtype=object)

In [11]:
submission = pd.DataFrame({
    "SampleID": df_test.SampleID,
    "label": y_final
})

submission

,SampleID,label
0,16001,sadness
1,16002,sadness
2,16003,sadness
3,16004,joy
4,16005,sadness
...,...,...
1995,17996,anger
1996,17997,anger
1997,17998,joy
1998,17999,joy


In [12]:
submission.to_csv("submission.csv", index=False)